# 03 Student Overfit — 8-object

In [ ]:
%matplotlib inline
import sys, os
sys.path.insert(0, os.path.abspath("../src"))
import matplotlib.pyplot as plt
from meshuv.data.dataset import CleanDataset
from meshuv.visualization import pipeline_views as V

DATASET = "processed/clean_v1"      # 相对 MESHUV_DATA_ROOT(env 可覆盖)
root = V.resolve(DATASET)
ds = CleanDataset(root, expose_diagnostics=True) if root else None
print("objects:", ds and len(ds))

In [ ]:
import json, numpy as np, torch
from meshuv.data.collate import collate
from meshuv.model.student_v0 import StudentV0
from meshuv.training.trainer import train
torch.manual_seed(3)
items = [ds[i] for i in range(min(8, len(ds)))]
batch = collate(items)
model = StudentV0()
r = train(model, batch, steps=800, lr=2e-3, log_every=200)
print(f"loss {r['loss_first']:.5f} -> {r['loss_last']:.6f}  "
      f"Spearman(active)={r['spearman_active']:.4f}")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 4))
axs[0].semilogy(r["losses"]); axs[0].set_title("loss")
m = batch["valid"]
axs[1].scatter(batch["target"][m], r["pred"][m], s=8, alpha=0.6)
axs[1].set_xlabel("target"); axs[1].set_ylabel("pred")
plt.tight_layout(); plt.show()

In [ ]:
# 逐对象: target vs prediction heatmap
OBJ = 0
item = items[OBJ]
a, b = batch["object_ranges"][OBJ]
fig = plt.figure(figsize=(13, 4.5))
V.show_target(item, fig.add_subplot(1, 3, 1))
V.show_prediction(item, r["pred"][a:b], fig.add_subplot(1, 3, 2))
V.show_charts(item, fig.add_subplot(1, 3, 3))
plt.tight_layout(); plt.show()